In [0]:
create table if not exists cloud_pricing.silver.aws_ec2_prices (
version string, version_ts timestamp, region_code string, instance_type string, instance_family string, os string,
vcpu int, memory_gib decimal(10,2), processor_architecture string, physical_processor string,
price_model string, term_years int, purchase_option string, offering_class string,
hourly_price decimal(20,10), upfront_fee decimal(20,10), effective_hourly decimal(20,6),
currency string, effective_date date, rate_code string, ingested_at timestamp
)

In [0]:
-- shared tenancy, plain linux/windows, no sql or other licence bundled, no capacity reservation / capacity block rows
-- this is the slice that's comparable with azure payg
create or replace temp view aws_rows as
select * from cloud_pricing.bronze.aws_ec2_prices_raw
where version not in (select distinct version from cloud_pricing.silver.aws_ec2_prices)
and product_family like 'Compute Instance%'
and tenancy = 'Shared' and capacitystatus = 'Used' and pre_installed_s_w = 'NA'
and license_model = 'No License required' and operating_system in ('Linux', 'Windows')
and marketoption = 'OnDemand' and currency = 'USD'

In [0]:
insert into cloud_pricing.silver.aws_ec2_prices
select
h.version, h.version_ts, h.region region_code, h.instance_type, h.instance_family, lower(h.operating_system) os,
cast(h.vcpu as int) vcpu, cast(regexp_extract(h.memory, '([0-9.]+)', 1) as decimal(10,2)) memory_gib,
h.processor_architecture, h.physical_processor,
case when h.termtype = 'OnDemand' then 'on_demand' else 'reserved' end price_model,
cast(regexp_extract(h.leasecontractlength, '([0-9]+)', 1) as int) term_years,
h.purchaseoption purchase_option, h.offeringclass offering_class,
cast(h.priceperunit as decimal(20,10)) hourly_price,
coalesce(cast(q.priceperunit as decimal(20,10)), 0) upfront_fee,
round(cast(h.priceperunit as decimal(20,10)) + coalesce(cast(q.priceperunit as decimal(20,10)), 0) / (coalesce(cast(regexp_extract(h.leasecontractlength, '([0-9]+)', 1) as int), 1) * 8760), 6) effective_hourly,
h.currency, to_date(h.effectivedate) effective_date, h.ratecode rate_code, h.ingested_at
from aws_rows h
-- reserved price is 2 rows, the hourly part (Hrs) and the upfront fee (Quantity), same sku + offer term
left join aws_rows q on q.sku = h.sku and q.offertermcode = h.offertermcode and q.version = h.version and q.region = h.region and q.unit = 'Quantity'
where h.unit = 'Hrs'

In [0]:
-- one price per instance/os/pricing option per version, fails otherwise
-- coalesce because count(distinct) skips rows with any null, on demand rows have no term
select version, region_code, assert_true(count(*) = count(distinct instance_type, os, price_model, coalesce(term_years, 0), coalesce(purchase_option, ''), coalesce(offering_class, '')), 'duplicate aws price rows')
from cloud_pricing.silver.aws_ec2_prices
group by version, region_code

In [0]:
select region_code, os, price_model, count(*) n, count(distinct instance_type) instances
from cloud_pricing.silver.aws_ec2_prices
group by all order by all